In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


In [ ]:
TELECOM_KNOWLEDGE = [

# ---------------- RAN: SCHEDULING ----------------
"Packet loss in uplink can be caused by inefficient scheduling algorithms, especially when proportional fairness is misconfigured under high load conditions.",
"High latency in downlink may result from scheduling delays when multiple users compete for limited radio resources in congested cells.",
"QoS-aware schedulers prioritize certain traffic classes, and misconfiguration can lead to starvation of lower-priority users.",

# ---------------- HARQ ----------------
"HARQ failures occur when retransmissions exceed maximum attempts, often due to poor channel conditions or incorrect link adaptation.",
"Frequent HARQ retransmissions indicate degraded radio conditions, which may be caused by interference or mobility issues.",
"Improper HARQ parameter tuning can increase latency and reduce throughput significantly.",

# ---------------- CQI / SINR ----------------
"Low CQI values typically indicate poor channel quality, often due to interference, distance from base station, or obstacles.",
"SINR degradation directly impacts throughput and increases block error rates in wireless systems.",
"High BLER suggests that modulation and coding schemes are not well adapted to current channel conditions.",

# ---------------- INTERFERENCE ----------------
"Inter-cell interference occurs when neighboring cells operate on overlapping frequencies, degrading signal quality.",
"Co-channel interference can significantly reduce SINR and lead to packet retransmissions.",
"Improper frequency reuse planning increases interference in dense deployments.",

# ---------------- CONGESTION ----------------
"Network congestion arises when user demand exceeds available radio or core network resources.",
"High traffic load during peak hours can cause packet drops and increased latency.",
"Buffer overflow in base stations leads to packet loss under heavy traffic conditions.",

# ---------------- POWER CONTROL ----------------
"Uplink power control issues can cause either excessive interference or insufficient signal strength at the base station.",
"Incorrect power settings may result in poor coverage or unnecessary interference with neighboring cells.",
"Dynamic power control is required to maintain optimal SINR levels across varying conditions.",

# ---------------- CORE NETWORK ----------------
"Packet loss in core networks can result from routing misconfigurations or overloaded gateways.",
"High latency in core networks may be caused by inefficient routing paths or processing delays in network functions.",
"Failure in network slicing mechanisms can degrade service quality for specific applications.",

# ---------------- HARDWARE ISSUES ----------------
"Hardware failures such as faulty antennas or damaged cables can degrade signal quality and cause packet loss.",
"Base station overheating may lead to performance throttling and increased latency.",
"Faulty RF components can result in unstable transmission conditions.",

# ---------------- SOFTWARE / CONFIG ----------------
"Misconfigured network parameters can lead to inefficient scheduling and degraded performance.",
"Software bugs in network controllers may cause unexpected packet drops or routing errors.",
"Incorrect firmware updates can introduce instability in network operations.",

# ---------------- MOBILITY ----------------
"Frequent handovers can increase packet loss if not properly managed.",
"Mobility issues arise when users move between cells with inconsistent signal quality.",
"Handover failures lead to temporary disconnections and degraded user experience.",

# ---------------- EDGE CASES ----------------
"Sudden spikes in traffic due to events can overload network resources.",
"Environmental factors such as weather can impact signal propagation.",
"Unexpected interference sources can degrade network performance unpredictably."

]


def convert_to_training_data(knowledge_list):
    data = []

    for item in knowledge_list:
        if "can be caused by" in item:
            parts = item.split("can be caused by")
            issue = parts[0].strip()
            cause = parts[1].strip()

        elif "may result from" in item:
            parts = item.split("may result from")
            issue = parts[0].strip()
            cause = parts[1].strip()

        elif "indicate" in item:
            parts = item.split("indicate")
            issue = parts[0].strip()
            cause = parts[1].strip()

        else:
            issue = "Network issue"
            cause = item

        data.append({
            "instruction": "Diagnose telecom network issue",
            "input": issue,
            "output": f"Root cause: {cause}. Recommended action: Investigate and optimize related network parameters."
        })

    return data


data = convert_to_training_data(TELECOM_KNOWLEDGE)

print(f"Dataset size: {len(data)}")
print(data[:3])

Dataset size: 33
[{'instruction': 'Diagnose telecom network issue', 'input': 'Packet loss in uplink', 'output': 'Root cause: inefficient scheduling algorithms, especially when proportional fairness is misconfigured under high load conditions.. Recommended action: Investigate and optimize related network parameters.'}, {'instruction': 'Diagnose telecom network issue', 'input': 'High latency in downlink', 'output': 'Root cause: scheduling delays when multiple users compete for limited radio resources in congested cells.. Recommended action: Investigate and optimize related network parameters.'}, {'instruction': 'Diagnose telecom network issue', 'input': 'Network issue', 'output': 'Root cause: QoS-aware schedulers prioritize certain traffic classes, and misconfiguration can lead to starvation of lower-priority users.. Recommended action: Investigate and optimize related network parameters.'}]


In [ ]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Output:
{example['output']}
"""
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # 🔥 CRITICAL FIX: add labels
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss
1,9.449883
2,8.619965
3,7.785307
4,6.842579
5,5.914845
6,4.824595
7,4.408872
8,3.724893
9,3.418660
10,2.868800


TrainOutput(global_step=85, training_loss=1.072104193533168, metrics={'train_runtime': 88.1236, 'train_samples_per_second': 1.872, 'train_steps_per_second': 0.965, 'total_flos': 704172942950400.0, 'train_loss': 1.072104193533168, 'epoch': 5.0})

In [ ]:
model.save_pretrained("telco-lora")
tokenizer.save_pretrained("telco-lora")

('telco-lora/tokenizer_config.json',
 'telco-lora/chat_template.jinja',
 'telco-lora/tokenizer.json')

In [ ]:
!zip -r telco-lora.zip telco-lora

  adding: telco-lora/ (stored 0%)
  adding: telco-lora/tokenizer_config.json (deflated 59%)
  adding: telco-lora/adapter_model.safetensors (deflated 8%)
  adding: telco-lora/README.md (deflated 65%)
  adding: telco-lora/chat_template.jinja (deflated 71%)
  adding: telco-lora/adapter_config.json (deflated 57%)
  adding: telco-lora/tokenizer.json (deflated 81%)


In [ ]:
from google.colab import files
files.download("telco-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>